# Analyse Exploratoire — Avantage Domicile en Premier League
**Projet Data Analytics — 2025/2026**

> Ce notebook présente l'analyse graphique de l'avantage domicile en Premier League sur 30 saisons (1993–2023). Les graphiques utilisent l'identité visuelle officielle de la Premier League.

---

## 0 — Setup
Chargement des librairies, du dataset et définition du thème visuel Premier League.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import os
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap
%matplotlib inline

# ── Détection automatique de la racine du projet ──────────────────────
# Remonte les dossiers parents jusqu'à trouver epl_strength.csv
# Fonctionne quel que soit le nom du dossier ou l'OS
def find_root(target='data/processed/epl_strength.csv', max_levels=6):
    current = Path().resolve()
    for _ in range(max_levels):
        if (current / target).exists():
            return current
        current = current.parent
    raise FileNotFoundError('Fichier non trouvé. Assurez-vous que le notebook est dans le repo.')

ROOT = find_root()
print('✓ Dossier racine détecté :', ROOT)

# ── Chargement du dataset ─────────────────────────────────────────────
df = pd.read_csv(ROOT / 'data' / 'processed' / 'epl_strength.csv')
os.makedirs(ROOT / 'outputs' / 'figures', exist_ok=True)

# ── Palette officielle Premier League ────────────────────────────────
PL = {
    'bg':     '#37003C',
    'green':  '#00FF87',
    'pink':   '#E90052',
    'purple': '#963CFF',
    'white':  '#FFFFFF',
    'grey':   '#7A7A8C',
    'grid':   '#FFFFFF15',
    'spine':  '#FFFFFF22',
}

def pl_theme(fig, axes, title, subtitle=None):
    fig.patch.set_facecolor(PL['bg'])
    for ax in axes:
        ax.set_facecolor(PL['bg'])
        ax.tick_params(colors=PL['white'], labelsize=9)
        ax.xaxis.label.set_color(PL['white'])
        ax.yaxis.label.set_color(PL['white'])
        for spine in ax.spines.values():
            spine.set_edgecolor(PL['spine'])
        ax.grid(True, color=PL['grid'], linewidth=0.5, linestyle='--')
        ax.set_axisbelow(True)
    fig.text(0.5, 0.97, title, ha='center', va='top',
             color=PL['white'], fontsize=13, fontweight='bold')
    if subtitle:
        fig.text(0.5, 0.91, subtitle, ha='center', va='top',
                 color=PL['grey'], fontsize=9)
    bar_ax = fig.add_axes([0, 0, 1, 0.018])
    bar_ax.set_facecolor(PL['pink'])
    bar_ax.set_axis_off()

def pl_legend(ax):
    return ax.legend(facecolor='#FFFFFF15', edgecolor='#FFFFFF30',
                     labelcolor=PL['white'], fontsize=9)

df['AwayWin'] = (df['FTR'] == 'A').astype(int)

print('✓ Setup OK — dataset chargé :', df.shape)
print('✓ Colonnes :', list(df.columns))

## 1 — Distribution de l'échantillon
**Objectif :** vérifier si notre variable cible (`HomeWin`) est équilibrée.

On s'attend à 3 classes équiprobables (33,3 % chacune) si le domicile ne jouait aucun rôle. Un déséquilibre en faveur des victoires à domicile serait une première indication d'un avantage structurel.

In [ ]:
result_counts = df['FTR'].value_counts().reindex(['H', 'D', 'A'])
result_pct    = result_counts / len(df) * 100

labels     = ['Victoire\nà domicile', 'Match\nnul', 'Victoire\nà l\'extérieur']
bar_colors = [PL['green'], PL['grey'], PL['pink']]

fig, ax = plt.subplots(figsize=(8, 5.5))
fig.patch.set_facecolor(PL['bg'])
ax.set_facecolor(PL['bg'])
fig.subplots_adjust(top=0.89, bottom=0.13, left=0.11, right=0.95)

bars = ax.bar(labels, result_pct.values, color=bar_colors, width=0.5,
              edgecolor=PL['bg'], linewidth=1.5)

for bar, val, count in zip(bars, result_pct.values, result_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.1f}%\n({count:,} matchs)',
            ha='center', va='bottom', color=PL['white'], fontweight='bold', fontsize=11)

ax.axhline(100/3, color=PL['white'], linestyle='--', linewidth=1.2, alpha=0.4)
ax.set_ylabel('% des matchs', color=PL['white'], fontsize=11)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 62)
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(colors=PL['white'], labelsize=10)

fig.text(0.5, 0.97, 'Distribution de l\'échantillon (1993–2023)',
         ha='center', va='top', color=PL['white'], fontsize=13, fontweight='bold')

ax.legend(
    handles=[plt.Line2D([0], [0], color=PL['white'], linestyle='--',
                        linewidth=1.5, label='Équilibre parfait (33.3%)')],
    facecolor=PL['bg'], edgecolor=PL['white'],
    labelcolor=PL['white'], fontsize=10, loc='upper right'
)

bar_ax = fig.add_axes([0, 0, 1, 0.018])
bar_ax.set_facecolor(PL['pink'])
bar_ax.set_axis_off()

plt.savefig(ROOT / 'outputs' / 'figures' / 'FIG_01_distribution_echantillon.png',
            dpi=150, bbox_inches='tight', facecolor=PL['bg'])
plt.show()

**Lecture :** Les victoires à domicile représentent **45,9 %** des matchs, soit environ **12 points de plus** que l'équilibre théorique (33,3 %). L'échantillon est donc déséquilibré en faveur du domicile — première indication d'un avantage structurel.

## 2 — Évolution du taux de victoire à domicile par saison
**Objectif :** observer si l'avantage domicile est stable dans le temps ou s'il varie.

On met en évidence les saisons Covid (2020–2021), jouées sans public, pour tester si la présence des supporters joue un rôle.

In [ ]:
season_stats = df.groupby('Season_End_Year').agg(
    HomeWinRate=('HomeWin', 'mean')
).reset_index()

global_mean = df['HomeWin'].mean() * 100

fig, ax = plt.subplots(figsize=(11, 5.5))
fig.patch.set_facecolor(PL['bg'])
ax.set_facecolor(PL['bg'])
fig.subplots_adjust(top=0.88, bottom=0.12, left=0.09, right=0.97)

ax.axvspan(2019.5, 2021.5, color=PL['pink'], alpha=0.12, zorder=0)

ax.fill_between(season_stats['Season_End_Year'],
                season_stats['HomeWinRate'] * 100, global_mean,
                where=season_stats['HomeWinRate'] * 100 >= global_mean,
                alpha=0.15, color=PL['green'], interpolate=True)
ax.fill_between(season_stats['Season_End_Year'],
                season_stats['HomeWinRate'] * 100, global_mean,
                where=season_stats['HomeWinRate'] * 100 < global_mean,
                alpha=0.15, color=PL['pink'], interpolate=True)

ax.axhline(global_mean, color=PL['white'], linestyle='--',
           linewidth=1.2, alpha=0.4, label=f'Moyenne sur 30 ans ({global_mean:.1f}%)')

ax.plot(season_stats['Season_End_Year'], season_stats['HomeWinRate'] * 100,
        color=PL['green'], linewidth=2.5, marker='o', markersize=5,
        markerfacecolor=PL['green'], markeredgecolor=PL['bg'], markeredgewidth=1.5,
        label='Taux de victoire à domicile', zorder=5)

ax.annotate('Saisons Covid\n(sans public)',
            xy=(2020.5, season_stats[season_stats['Season_End_Year'] == 2020]['HomeWinRate'].values[0] * 100),
            xytext=(2018.2, 58), color=PL['pink'], fontsize=9, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=PL['pink'], lw=1.2, connectionstyle='arc3,rad=-0.2'))

ax.set_xlabel('Saison (année de fin)', color=PL['white'], fontsize=10)
ax.set_ylabel('% de victoires à domicile', color=PL['white'], fontsize=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(28, 68)
ax.tick_params(colors=PL['white'], labelsize=9)
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color(PL['white'])
ax.spines['left'].set_color(PL['white'])

fig.text(0.5, 0.97, 'Évolution du taux de victoire à domicile par saison (1993–2023)',
         ha='center', va='top', color=PL['white'], fontsize=13, fontweight='bold')

ax.legend(facecolor=PL['bg'], edgecolor=PL['white'],
          labelcolor=PL['white'], fontsize=9, loc='upper right')

bar_ax = fig.add_axes([0, 0, 1, 0.018])
bar_ax.set_facecolor(PL['pink'])
bar_ax.set_axis_off()

plt.savefig(ROOT / 'outputs' / 'figures' / 'FIG_03_evolution_homewin.png',
            dpi=150, bbox_inches='tight', facecolor=PL['bg'])
plt.show()

**Lecture :** Le taux de victoire à domicile oscille autour de **45–47 %** sur l'ensemble de la période, ce qui confirme sa stabilité structurelle. Les saisons Covid (zone rose) marquent un **creux visible**, suggérant que l'absence de public réduit l'avantage domicile.